# Example notebook for UMAP / t-SNE dimensionality reduction plots

The present notebook serves as a guide of how to use the `IDEAL-GENOM-QC` library to draw and analyse UMAP and t-SNE plots of population structure.

The underlying module (`ideal_genom.population.projection`) splits this into composable pieces — `PCAReduction` (LD pruning + PCA), `UMAPReduction`, `TSNEReduction`, and `Plot2D` for metadata-aware plotting — all orchestrated by `DimensionalityReductionPipeline`. We show both a single-parameter run and a parameter-grid sweep (the grid is auto-detected whenever a parameter value is a list instead of a scalar).

Let us import the required libraries.

In [ ]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.population.projection import DimensionalityReductionPipeline

In the next cell the path variables associated with the project are set.

As with ancestry QC, this step is typically run on the cleaned output of the sample QC pipeline.

In [ ]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
ouputData = test_data / 'outputData'

# Use the cleaned output of the sample QC notebook as input
input_path = ouputData / 'sample_qc_results' / 'clean_files'
input_name = '1KG_GRCh38_sample_qc'
output_path = ouputData
high_ld_file = Path('path/to/ld_file') # if not available, set to a non-existent Path() and it will be fetched automatically

In the next cell we define the parameter dictionaries used by the pipeline.

**PCA preparation** (`pca_params`, used for LD pruning + PCA, required before either reduction):

1. `maf`: Parameter `--maf` of **PLINK1.9**.
2. `mind`: Parameter `--mind` of **PLINK1.9**.
3. `geno`: Parameter `--geno` of **PLINK1.9**.
4. `hwe`: Parameter `--hwe` of **PLINK1.9**.
5. `ind_pair`: Parameter `--indep-pairwise` of **PLINK1.9**.
6. `pca`: number of principal components computed; these are the components UMAP/t-SNE will be run on.

**UMAP** (`umap_params`) and **t-SNE** (`tsne_params`): these accept either single scalar values (one run) or lists of values (automatically triggers a grid search over all combinations — see `umap-learn` docs at https://umap-learn.readthedocs.io/en/latest/ and scikit-learn's t-SNE docs for parameter meanings).

**Plotting**:

- `color_hue_file`: optional path to a tab-separated file whose first two columns match the `.fam` file's ID columns, and whose third column is a categorical variable used as plot hue (e.g. the `population_tags` file produced by the ancestry QC notebook).
- `case_control_marker`: if `True`, uses the `.fam` file's case/control phenotype as the plot hue/style instead.

In [ ]:
pca_params = {
    'maf': 0.01,
    'mind': 0.2,
    'geno': 0.1,
    'hwe': 5e-8,
    'ind_pair': [20000, 2000, 0.2],
    'pca': 25,
}

# Single-run parameters
umap_params_single = {
    'n_neighbors': 15,
    'min_dist': 0.1,
    'metric': 'euclidean',
    'random_state': 42,
}

tsne_params_single = {
    'perplexity': 30.0,
    'random_state': 42,
}

# Set to a Path, e.g. the `population_tags` file from the ancestry QC notebook, if you want population-colored plots
color_hue_file = None
case_control_marker = False

Initialize the `DimensionalityReductionPipeline`.

In [ ]:
pipeline = DimensionalityReductionPipeline(
    input_path =input_path,
    input_name =input_name,
    output_path=output_path,
    build      ='38',
    high_ld_regions_file=high_ld_file,
)

First, run a single-parameter pass: PCA preparation, then one UMAP run and one t-SNE run, then the corresponding plots. `execute_dimensionality_reduction_pipeline()` auto-detects that none of the `umap_params`/`tsne_params` values are lists, so it takes the single-run path.

The PCA preparation step shells out to PLINK, which prints a lot of console text; we capture it into `dimred_single_log` to keep the notebook readable — run `dimred_single_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture dimred_single_log
results_single = pipeline.execute_dimensionality_reduction_pipeline(
    pca_params=pca_params,
    run_umap=True,
    umap_params=umap_params_single,
    run_tsne=True,
    tsne_params=tsne_params_single,
    color_hue_file=color_hue_file,
    case_control_markers=case_control_marker,
    plot_format='svg',
)

In [ ]:
print("Single-parameter UMAP/t-SNE run completed.")

`results_single` is a summary dict pointing at every file produced (eigenvector/eigenvalue, UMAP/t-SNE coordinates, plots).

Note: the PCA preparation step (Step 1) also generates its own 2D scatter plot (PC1 vs PC2) as a side effect, but it isn't tracked in `results_single['files']` — it's saved separately, as `pca_2d_plot.pdf`, directly under `pipeline.results_dir` (not the `plots/` subfolder used by UMAP/t-SNE, and always PDF regardless of `plot_format`).

In [ ]:
results_single['files']

In [ ]:
pca_plot_path = pipeline.results_dir / 'pca_2d_plot.pdf'
print(f"PCA scatter plot (PC1 vs PC2): {pca_plot_path} (exists: {pca_plot_path.exists()})")

The UMAP and t-SNE coordinates are also kept in memory on the pipeline object, so they can be inspected directly without re-reading from disk.

Now let's explore the parameter space. Passing **lists** instead of scalars for `umap_params`/`tsne_params` makes `execute_dimensionality_reduction_pipeline()` automatically switch to a grid search over every combination (via `execute_parameter_grid()` internally). Since `force_pca_recompute=False` by default, the PCA computed above is reused — only the UMAP/t-SNE step is repeated for each combination.

We capture this cell's output into `dimred_grid_log` to keep the notebook readable — run `dimred_grid_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture dimred_grid_log
umap_grid_params = {
    'n_neighbors': [5, 10, 15, 20, 25, 30],
    'metric': ['euclidean', 'chebyshev'],
    'min_dist': [0.01, 0.1, 0.2],
    'random_state': [42],
}

tsne_grid_params = {
    'perplexity': [20.0, 30.0, 50.0],
    'random_state': [42],
}

results_grid = pipeline.execute_dimensionality_reduction_pipeline(
    pca_params=pca_params,
    run_umap=True,
    umap_params=umap_grid_params,
    run_tsne=True,
    tsne_params=tsne_grid_params,
    color_hue_file=color_hue_file,
    case_control_markers=case_control_marker,
    plot_format='svg',
)

In [ ]:
print(f"Parameter grid search completed. Summary written to: {pipeline.results_dir / 'parameter_grid_summary.tsv'}")

The grid search writes a `parameter_grid_summary.tsv` with one row per combination explored (for both UMAP and t-SNE), which we can load to compare runs.

In [ ]:
grid_summary = pd.read_csv(pipeline.results_dir / 'parameter_grid_summary.tsv', sep='\t')
grid_summary

Coordinates and plots for every combination are saved under `umap_grid_results/` and `tsne_grid_results/` inside the pipeline's results directory, named after their parameters (e.g. `umap_n_neighbors15_metriceuclidean_min_dist0.1_plot.svg`). Let's list what was generated for UMAP.

In [ ]:
sorted(p.name for p in (pipeline.results_dir / 'umap_grid_results').glob('*_plot.svg'))

Note: unlike sample/variant/ancestry QC, there is no dedicated clean-up class for this module — intermediate PLINK files from the LD pruning + PCA step remain in `pipeline.results_dir` for inspection.